In [7]:
import json
from pathlib import Path

import pandas as pd
import requests

POSTCODES_API = "https://api.postcodes.io"

# Walk up from the current folder until we find the one containing data/raw
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "london_2026-09-22.csv"

print("Project root:", PROJECT_ROOT)
print("Reading from:", RAW_FILE)
print("File exists: ", RAW_FILE.exists())

Project root: c:\projects\food-hygiene-risk-london
Reading from: c:\projects\food-hygiene-risk-london\data\raw\london_2026-09-22.csv
File exists:  True


In [8]:
# Look up one postcode

response = requests.get(f"{POSTCODES_API}/postcodes/SW1A 1AA", timeout=30)
print("Status code:", response.status_code)

result = response.json()["result"]
print("\nTop-level fields:", list(result.keys()))
print("\ncodes section:")
print(json.dumps(result["codes"], indent=2))

Status code: 200

Top-level fields: ['postcode', 'quality', 'eastings', 'northings', 'country', 'nhs_ha', 'longitude', 'latitude', 'european_electoral_region', 'primary_care_trust', 'region', 'lsoa', 'msoa', 'incode', 'outcode', 'parliamentary_constituency', 'parliamentary_constituency_2024', 'senedd_constituency', 'senedd_constituency_no', 'admin_district', 'parish', 'admin_county', 'date_of_introduction', 'date_of_termination', 'index_of_multiple_deprivation', 'admin_ward', 'ced', 'ccg', 'nuts', 'pfa', 'nhs_region', 'ttwa', 'national_park', 'bua', 'icb', 'cancer_alliance', 'lsoa11', 'msoa11', 'lsoa21', 'msoa21', 'oa21', 'ruc11', 'ruc21', 'lep1', 'lep2', 'codes']

codes section:
{
  "admin_district": "E09000033",
  "admin_county": "E99999999",
  "admin_ward": "E05013806",
  "parish": "E43000236",
  "parliamentary_constituency": "E14001172",
  "parliamentary_constituency_2024": "E14001172",
  "ccg": "E38000256",
  "ccg_id": "W2U3Z",
  "ced": "E99999999",
  "nuts": "TLI35",
  "lsoa": "E

In [9]:
# How messsy are postcodes
df = pd.read_csv(RAW_FILE)

postcodes = df["PostCode"]
print("Total businesses:     ", len(postcodes))
print("Missing postcode:     ", postcodes.isna().sum())
print("Unique postcodes:     ", postcodes.nunique())

clean = postcodes.str.strip().str.upper()
short = clean[clean.str.len() < 5].value_counts()
print("\nPartial postcodes (under 5 characters):", short.sum())
print(short.head(10))

Total businesses:      81572
Missing postcode:      974
Unique postcodes:      29115

Partial postcodes (under 5 characters): 9744
PostCode
CR0     276
SE18    128
UB6     123
HA8     107
UB5     106
IG1     105
HA3     101
SW16    100
SE6     100
IG11     96
Name: count, dtype: int64


In [10]:
# A small bulk request, including a bad postcode on purpose

test_batch = ["SW1A 1AA", "W1U", "EC1A 1BB", "NOT A POSTCODE"]

response = requests.post(
    f"{POSTCODES_API}/postcodes",
    json={"postcodes": test_batch},
    timeout=30,
)
print("Status code:", response.status_code)

for item in response.json()["result"]:
    found = item["result"] is not None
    print(f"{item['query']:<16} found={found}")

Status code: 200
SW1A 1AA         found=True
W1U              found=False
EC1A 1BB         found=True
NOT A POSTCODE   found=False


In [11]:
# Checking whether it matters

rated = df[df["RatingValue"].isin(["0", "1", "2", "3", "4", "5"])].copy()
rated["fail"] = rated["RatingValue"].astype(int) <= 2

pc = rated["PostCode"].str.strip().str.upper()
rated["postcode_status"] = "full"
rated.loc[pc.str.len() < 5, "postcode_status"] = "partial"
rated.loc[pc.isna(), "postcode_status"] = "missing"

summary = rated.groupby("postcode_status")["fail"].agg(businesses="size", fail_rate="mean")
summary["fail_rate"] = (summary["fail_rate"] * 100).round(1).astype(str) + "%"
print(summary)

                 businesses fail_rate
postcode_status                      
full                  63865      6.1%
missing                 827      2.4%
partial                6698      1.3%


## Postcode exploration: findings
- postcodes.io returns several LSOA fields. We use **`codes.lsoa21`** because
  IoD2025 uses 2021 LSOA boundaries (some London LSOAs changed between 2011 and 2021).
- The bulk endpoint (POST, up to 100 postcodes) returns `result = None` for bad or
  partial postcodes instead of raising an error.
- 29,115 unique postcodes, so about 290 bulk requests.
- Postcode status (all businesses): 974 missing, 9,744 partial (district only, address withheld).
- Fail rate by status (rated businesses only): full 6.1%, missing 2.4%, **partial 1.3%**.
- Withheld-address businesses fail about 4.7x less often than the rest, so we keep them
  and add an `address_withheld` flag. To check in EDA: is this flag just standing in for business type?
- Fallback plan for area features: LSOA (full postcode), then postcode district average
  (partial), then borough average (missing).